In [2]:
import os
import random
import time
from typing import Any, Dict, List

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model
import torch
import math
import matplotlib.pyplot as plt

    
SEED = 42


ImportError: huggingface-hub>=0.34.0,<1.0 is required for a normal functioning of this module, but found huggingface-hub==1.1.7.
Try: `pip install transformers -U` or `pip install -e '.[dev]'` if you're working with git main

In [20]:
dataset = load_dataset("yahma/alpaca-cleaned")
print(dataset)


DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 51760
    })
})


In [22]:
random.seed(SEED)
full_train = dataset["train"].shuffle(seed=SEED)

train_data = full_train.select(range(0, 10000))
val_data   = full_train.select(range(10000, 12000))
test_data  = full_train.select(range(12000, 14000))

print("Train size:", len(train_data))
print("Validation size:", len(val_data))
print("Test size:", len(test_data))


Train size: 10000
Validation size: 2000
Test size: 2000


In [23]:
def format_example(example: Dict[str, Any]) -> Dict[str, str]:
    instruction = example["instruction"]
    input_text  = example["input"]
    output_text = example["output"]

    if input_text:
        prompt = f"Instruction: {instruction}\nInput: {input_text}\nResponse:"
    else:
        prompt = f"Instruction: {instruction}\nResponse:"

    return {
        "prompt": prompt,
        "label": output_text,
    }

train_data = train_data.map(format_example)
val_data   = val_data.map(format_example)
test_data  = test_data.map(format_example)

print("Example formatted prompt:")
print(train_data[0]["prompt"])
print("-----")
print("Label:")
print(train_data[0]["label"])

Example formatted prompt:
Instruction: Rearrange the following sentence to make the sentence more interesting.
Input: She left the party early
Response:
-----
Label:
Early, she left the party.


In [25]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-3.2-1B",
    
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

Loading tokenizer...


In [26]:
def tokenize_function(batch: Dict[str, List[str]]) -> Dict[str, Any]:
    texts = [p + " " + l for p, l in zip(batch["prompt"], batch["label"])]
    return tokenizer(
        texts,
        max_length=512,
        truncation=True,
        padding=False,  # dynamic padding in collator
    )

print("Tokenizing train...")
train_tokenized = train_data.map(
    tokenize_function,
    batched=True,
    remove_columns=train_data.column_names,
)

print("Tokenizing validation...")
val_tokenized = val_data.map(
    tokenize_function,
    batched=True,
    remove_columns=val_data.column_names,
)

print("Tokenizing test...")
test_tokenized = test_data.map(
    tokenize_function,
    batched=True,
    remove_columns=test_data.column_names,
)

print("Example tokenized sample:")
print({k: v[:10] for k, v in train_tokenized[0].items()})

Tokenizing train...
Tokenizing validation...


Map: 100%|██████████| 2000/2000 [00:00<00:00, 7099.01 examples/s]


Tokenizing test...
Example tokenized sample:
{'input_ids': [128000, 17077, 25, 47002, 9866, 279, 2768, 11914, 311, 1304], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [28]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

In [29]:
print("Loading Llama-3.2-1B model...")
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-1B",
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

model.config.pad_token_id = tokenizer.pad_token_id

lora_config = LoraConfig(
    r=8,              # rank (to analyse in report)
    lora_alpha=16,
    lora_dropout=0.1,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
print("LoRA model ready.")

Loading Llama-3.2-1B model...
LoRA model ready.


In [38]:
training_args = TrainingArguments(
    output_dir="./outputs/checkpoints",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    logging_steps=50, 
    eval_strategy="steps", 
    eval_steps=200, 
    save_strategy = "steps", 
    save_steps= 200, 
    save_total_limit=3, 
    num_train_epochs=3,  # change to 3 for full run if you want
    fp16= True if torch.cuda.is_available() else False,
    seed=SEED,
    
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    tokenizer=tokenizer,
)


/tmp/ipykernel_543810/2750528672.py:21: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [39]:
start_time = time.time()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

print("Starting training...")
trainer.train()
end_time = time.time()
total_time_sec = end_time - start_time
print(f"Total training time: {total_time_sec:.2f} seconds")

if torch.cuda.is_available():
    peak_mem_bytes = torch.cuda.max_memory_allocated()
    peak_mem_gb = peak_mem_bytes / (1024 ** 3)
    print(f"Peak GPU memory usage: {peak_mem_gb:.2f} GB")

Starting training...


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [15]:
eval_metrics = trainer.evaluate(eval_dataset=val_tokenized)
print("Validation metrics:", eval_metrics)

if "eval_loss" in eval_metrics:
    try:
        ppl = math.exp(eval_metrics["eval_loss"])
        print(f"Validation perplexity: {ppl:.4f}")
    except OverflowError:
        print("Validation perplexity: overflow (loss too large)")

Validation metrics: {'eval_loss': 1.4205402135849, 'eval_runtime': 31.951, 'eval_samples_per_second': 62.596, 'eval_steps_per_second': 15.649, 'epoch': 1.0}
Validation perplexity: 4.1394


In [17]:
os.makedirs("./outputs/plots", exist_ok=True)

train_steps = []
train_losses = []
eval_steps = []
eval_losses = []

for entry in trainer.state.log_history:
    # training loss
    if "loss" in entry and "step" in entry:
        train_steps.append(entry["step"])
        train_losses.append(entry["loss"])
    # validation loss (from any evaluate() call)
    if "eval_loss" in entry and "step" in entry:
        eval_steps.append(entry["step"])
        eval_losses.append(entry["eval_loss"])

plt.figure()
if train_losses:
    plt.plot(train_steps, train_losses, label="train_loss")
if eval_losses:
    plt.plot(eval_steps, eval_losses, label="eval_loss")

plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid(True)

plot_path = "./outputs/plots/loss_curves.png"
plt.savefig(plot_path, bbox_inches="tight")
plt.close()
print(f"Saved loss curves to {plot_path}")

Saved loss curves to ./outputs/plots/loss_curves.png


In [ ]:
os.makedirs("./outputs", exist_ok=True)
trainer.save_model("./outputs/best_model.pt")
tokenizer.save_pretrained("./outputs/checkpoints")
print("Training finished and model saved to ./outputs/best_model.pt")